# Building Footprint Detection

This notebook extracts building footprints from a high-resolution aerial image using the pre-trained Mask R-CNN model in [`geoai-py`](https://github.com/opengeos/geoai).

Since the input image can be very large, and higher resolution than the model was trained on, the pipeline:

1. Splits the image into tiles
2. Reprojects each tile to a local UTM CRS (so resolution is in meters)
3. Resamples tiles to ~50cm if the source is higher resolution than the model's 50cm-1m training range
4. Runs the building footprint model on each tile
5. Mosaics the resulting masks
6. Vectorizes the masks and adds geometric properties
7. Visualizes the results

Structure adapted from the GeoAI [Building Footprint Extraction for Africa](https://opengeoai.org/examples/building_footprints_africa/) example.

## Install package

To use the `geoai-py` package, ensure it is installed in your environment. Uncomment the command below if needed.

In [ ]:
%pip install -q geoai-py rioxarray

## Import libraries

In [2]:
import os
import glob

import rasterio
from rasterio.warp import calculate_default_transform
from rasterio.enums import Resampling
import rioxarray

import geoai

## Set up working directories

In [3]:
work_dir = "building_detection"
tiles_dir = os.path.join(work_dir, "tiles")
processed_dir = os.path.join(work_dir, "processed")
masks_dir = os.path.join(work_dir, "masks")

for d in [work_dir, tiles_dir, processed_dir, masks_dir]:
    os.makedirs(d, exist_ok=True)

## Download the input image

Using a sample high-resolution aerial image from OpenAerialMap.

In [ ]:
raster_url = "https://oin-hotosm-temp.s3.us-east-1.amazonaws.com/69493c8084a859b011c94266/0/69493c8084a859b011c94267.tif"
raster_path = geoai.download_file(raster_url, output_path=os.path.join(work_dir, "input.tif"))

## Inspect the image resolution and CRS

Reading only metadata (no pixel stats) keeps this fast even for very large rasters.

In [5]:
with rasterio.open(raster_path) as src:
    src_crs = src.crs
    src_width, src_height = src.width, src.height
    src_bounds = src.bounds
    src_res = src.res

print(f"CRS: {src_crs}")
print(f"Size: {src_width} x {src_height} pixels")
print(f"Native pixel size: {src_res} ({'degrees' if src_crs.is_geographic else 'meters'})")

CRS: EPSG:4326
Size: 43459 x 38243 pixels
Native pixel size: (2.796960705804071e-07, 2.796960705804071e-07) (degrees)


## Determine the local UTM CRS

The model expects resolution in meters, so reproject to the UTM zone matching the image's location.

In [6]:
def get_utm_crs(lon, lat):
    zone = int((lon + 180) / 6) + 1
    epsg = 32600 + zone if lat >= 0 else 32700 + zone
    return f"EPSG:{epsg}"


centroid_lon = (src_bounds.left + src_bounds.right) / 2
centroid_lat = (src_bounds.bottom + src_bounds.top) / 2
utm_crs = get_utm_crs(centroid_lon, centroid_lat)
print(f"Local UTM CRS: {utm_crs}")

Local UTM CRS: EPSG:32643


## Check whether resampling is needed

The building footprint model is trained on 50cm-1m resolution imagery. If the source image is higher resolution (e.g. 5-10cm), tiles will be resampled down to 50cm before inference.

In [7]:
target_resolution = 0.5  # meters - within the model's 50cm-1m training range

transform, _, _ = calculate_default_transform(
    src_crs, utm_crs, src_width, src_height, *src_bounds
)
native_resolution_m = abs(transform.a)
print(f"Native ground resolution: {native_resolution_m:.3f} m")

resample_needed = native_resolution_m < target_resolution
if resample_needed:
    print(
        f"Higher resolution than {target_resolution} m -> tiles will be resampled to {target_resolution} m"
    )
else:
    print("Resolution already within the model's training range -> no resampling needed")

Native ground resolution: 0.030 m
Higher resolution than 0.5 m -> tiles will be resampled to 0.5 m


## Split the image into tiles

The source image can be too large to reproject and process at once, so it is first split into georeferenced tiles.

In [8]:
tile_size = 8192  # pixels, at native resolution (tune based on image size/available memory)
stride = tile_size  # no overlap between tiles

geoai.export_geotiff_tiles(
    in_raster=raster_path,
    out_folder=tiles_dir,
    tile_size=tile_size,
    stride=stride,
)

tile_paths = sorted(glob.glob(os.path.join(tiles_dir, "images", "*.tif")))
print(f"Generated {len(tile_paths)} tiles")

Generating tiles:   0%|          | 0/30 [00:00<?, ?it/s]

Generated 30 tiles


## Reproject and resample tiles to the local UTM CRS

In [9]:
def reproject_tile(tile_path, out_path, dst_crs, resolution=None):
    da = rioxarray.open_rasterio(tile_path)
    kwargs = {"resampling": Resampling.bilinear}
    if resolution is not None:
        kwargs["resolution"] = (resolution, resolution)
    # Force clean RGB tagging - the source tiles inherit the original
    # image's photometric/compression tags (often YCbCr+JPEG for aerial
    # imagery), which can corrupt rendering even though pixel values
    # decode correctly.
    da.rio.reproject(dst_crs, **kwargs).rio.to_raster(
        out_path, photometric="RGB", compress="LZW"
    )
    return out_path


processed_paths = []
for tile_path in tile_paths:
    out_path = os.path.join(processed_dir, os.path.basename(tile_path))
    reproject_tile(
        tile_path,
        out_path,
        dst_crs=utm_crs,
        resolution=target_resolution if resample_needed else None,
    )
    processed_paths.append(out_path)

print(f"Reprojected {len(processed_paths)} tiles to {utm_crs}")

Reprojected 30 tiles to EPSG:32643


## Initialize the building footprint extraction model

In [ ]:
extractor = geoai.BuildingFootprintExtractor(model_path="building_footprints_usa.pth")

## Run inference on each tile

In [11]:
for tile_path in processed_paths:
    mask_path = os.path.join(masks_dir, os.path.basename(tile_path))
    extractor.generate_masks(
        tile_path,
        output_path=mask_path,
        min_object_area=50,
        confidence_threshold=0.5,
        mask_threshold=0.5,
    )

print("Finished generating masks for all tiles")

Finished generating masks for all tiles


## Mosaic the tile masks

In [12]:
masks_mosaic_path = os.path.join(work_dir, "building_masks.tif")
geoai.mosaic_geotiffs(masks_dir, masks_mosaic_path)

True

## Mosaic the image tiles

Mosaic the reprojected/resampled RGB tiles so the actual imagery (in the same UTM CRS as the results) can be used as the background layer for comparison.

In [13]:
image_mosaic_path = os.path.join(work_dir, "image_mosaic.tif")
geoai.mosaic_geotiffs(processed_dir, image_mosaic_path)

True

## Visualize the mask

In [ ]:
geoai.view_raster(masks_mosaic_path, opacity=0.7, colormap="tab20", basemap=raster_url)

## Vectorize the building footprints

In [ ]:
gdf = geoai.orthogonalize(
    input_path=masks_mosaic_path,
    output_path=os.path.join(work_dir, "building_footprints.geojson"),
    epsilon=1.0,
)

## Add geometric attributes

In [ ]:
gdf = geoai.add_geometric_properties(gdf)
gdf.head()

## Visualize results

In [17]:
geoai.view_vector_interactive(
    gdf, style_kwds={"color": "red", "fillOpacity": 0.2}, tiles=raster_url
)